In [ ]:
FW_DIR = "../fw/examples/cw305_driver_example"
BIN_FILE = f"{FW_DIR}/cw305_driver_example.bin"
BOOTLOADER = "python3 ../../sdk/toolchain/bootloader.py"
BUFF_LEN = 22
BS_FILE = "../vivado/risqrypt_cw305.runs/impl_1/fpga_top.bit"
VERBOSE = True

In [ ]:
import chipwhisperer as cw
scope = cw.scope()
scope.adc.offset = 0
scope.adc.basic_mode = "rising_edge"
scope.trigger.triggers = "tio4"
scope.io.tio1 = "serial_rx"
scope.io.tio2 = "serial_tx"
scope.io.hs2 = "disabled"

In [ ]:
TARGET_PLATFORM = 'CW305_100t'

In [ ]:
scope.gain.db = 25
platform = 'cw305'
fpga_id = '100t'

target = cw.target(scope, cw.targets.CW305, force=False, fpga_id=fpga_id, platform=platform, bsfile=BS_FILE)

In [ ]:
target.vccint_set(1.0)
# we only need PLL1:
target.pll.pll_enable_set(True)
target.pll.pll_outenable_set(False, 0)
target.pll.pll_outenable_set(True, 1)
target.pll.pll_outenable_set(False, 2)

# run at 10 MHz:
target.pll.pll_outfreq_set(20E6, 1)

# 1ms is plenty of idling time
target.clkusbautooff = True
target.clksleeptime = 1

In [ ]:
%%bash -s "$FW_DIR" "$BUFF_LEN"
cd $1
make USE_DONE=1 BUFF_LEN=$2

In [ ]:
%%bash -s "$BIN_FILE" "$BOOTLOADER"
$2 -f $1 -q

In [ ]:
import time
time.sleep(3)

In [ ]:
import tqdm
import random

hex_len = BUFF_LEN

for i in tqdm.tqdm(range(128)):
    message = bytes([random.randint(0, 255) for _ in range(hex_len//2 + 1)]).hex()[:hex_len]
    if VERBOSE:
        print("Sending message {}".format(message))
    target.fpga_write(0, message)
    while(True):
        status = target.fpga_read(1, 1)
        if status[0] == 0xff:
            break
    s = target.fpga_read(0, len(message))
    if VERBOSE:
        print(s)
    s_ = s.decode('utf-8')
    assert s_ == message, "Mismatch: sent {}, received {}".format(message, s_)
    if VERBOSE:
        print("Read back: {}".format(s.decode('utf-8')))
        print()
        print()

In [ ]:
if scope._is_husky:
    scope.clock.clkgen_freq = 40e6
    scope.clock.clkgen_src = 'extclk'
    scope.clock.adc_mul = 2
    # if the target PLL frequency is changed, the above must also be changed accordingly
else:
    scope.clock.adc_src = "extclk_x4"

In [ ]:
import time
for i in range(5):
    scope.clock.reset_adc()
    time.sleep(1)
    if scope.clock.adc_locked:
        break 
assert (scope.clock.adc_locked), "ADC failed to lock"

In [ ]:
%%bash -s "$FW_DIR" "$BUFF_LEN"
cd $1
make BUFF_LEN=$2

In [ ]:
%%bash -s "$BIN_FILE" "$BOOTLOADER"
$2 -f $1 -q

In [ ]:
import time
time.sleep(3)

In [ ]:
import tqdm
import random

hex_len = BUFF_LEN

for i in tqdm.tqdm(range(128)):
    message = bytes([random.randint(0, 255) for _ in range(hex_len//2 + 1)]).hex()[:hex_len]
    if VERBOSE:
        print("Sending message {}".format(message))
    scope.arm()
    if VERBOSE:
        print("Sending message {}".format(message))
    target.fpga_write(0, message)
    ret = scope.capture(poll_done=True)
    s = target.fpga_read(0, len(message))
    if VERBOSE:
        print(s)
    s_ = s.decode('utf-8')
    assert s_ == message, "Mismatch: sent {}, received {}".format(message, s_)
    if VERBOSE:
        print("Read back: {}".format(s.decode('utf-8')))
        print()
        print()